# Silver → Gold: Aggregation Exploration
## Gold Layer Design Notebook

This notebook explores the aggregation logic required to transform Silver Parquet files
into a single Gold dataset combining the three Banxico financial series.

Silver input:
- `silver_tipo_de_cambio` — daily USD/MXN FIX exchange rate
- `silver_tiie_28`        — daily TIIE 28-day interbank rate
- `silver_inpc`           — monthly INPC consumer price index

Gold output:
- One unified monthly dataset combining all three series with derived metrics
- Partitioned by year/month, written as Parquet to S3

Key questions:
1. How do we aggregate daily series to monthly?
2. What derived metrics are meaningful for a fintech dashboard?
3. How do we handle the INPC lag (38 months vs 40 for daily series)?
4. What does the final Gold schema look like?

In [105]:
import io
import os

import boto3
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

BUCKET_NAME = os.getenv("BUCKET_NAME", "banxico-pipeline-dev-datalake")
AWS_REGION  = os.getenv("AWS_REGION",  "us-east-1")

SERIES = {
    "tipo_de_cambio": {"id": "SF43718", "frequency": "daily"},
    "tiie_28":        {"id": "SF60648", "frequency": "daily"},
    "inpc":           {"id": "SP1",     "frequency": "monthly"},
}

s3_client = boto3.client("s3", region_name=AWS_REGION)

print(f"Bucket : {BUCKET_NAME}")
print(f"Region : {AWS_REGION}")

Bucket : banxico-pipeline-dev-datalake
Region : us-east-1


## 1. Read Silver Datasets

Read all Parquet partitions for each dataset from S3 Silver layer.
Uses s3fs so pandas can read S3 paths directly.

In [106]:
def read_silver_dataset(dataset: str) -> pd.DataFrame:
    """
    Read all Silver Parquet partitions for a given dataset from S3.

    Lists all year/month partitions and concatenates them into a single
    DataFrame sorted by date.
    """
    prefix   = f"silver/source=banxico/dataset={dataset}/"
    response = s3_client.list_objects_v2(Bucket=BUCKET_NAME, Prefix=prefix)
    keys     = [
        obj["Key"]
        for obj in response.get("Contents", [])
        if obj["Key"].endswith(".parquet")
    ]

    if not keys:
        print(f"WARNING: No Parquet files found for dataset={dataset}")
        return pd.DataFrame()

    dfs = []
    for key in keys:
        obj    = s3_client.get_object(Bucket=BUCKET_NAME, Key=key)
        buffer = io.BytesIO(obj["Body"].read())
        dfs.append(pd.read_parquet(buffer))

    df = pd.concat(dfs, ignore_index=True)
    df["date"] = pd.to_datetime(df["date"])
    return df.sort_values("date").reset_index(drop=True)

## 2. Inspect Silver Data Quality

Before aggregating, verify that Silver data is clean and complete.
Any issues here will propagate to Gold.

In [107]:
def inspect_df(df: pd.DataFrame, name: str = "DataFrame") -> None:
    """Print a concise quality summary for a DataFrame indexed by date."""
    print(f"\n── {name} ──")
    print(f"Shape     : {df.shape}")
    print(f"Dtypes    :\n{df.dtypes}")
    print(f"Date range: {df.index.min().date()} → {df.index.max().date()}")
    print(f"Nulls     :\n{df.isnull().sum()}")

    dupes = df[df.index.duplicated(keep=False)]
    print(f"Duplicates: {len(dupes)}")
    if not dupes.empty:
        print(dupes)


# Load raw Silver for inspection
fx_raw   = read_silver_dataset("tipo_de_cambio").set_index("date")
tiie_raw = read_silver_dataset("tiie_28").set_index("date")
inpc_raw = read_silver_dataset("inpc").set_index("date")

inspect_df(fx_raw,   "Silver — tipo_de_cambio")
inspect_df(tiie_raw, "Silver — tiie_28")
inspect_df(inpc_raw, "Silver — inpc")


── Silver — tipo_de_cambio ──
Shape     : (818, 6)
Dtypes    :
source           object
dataset          object
serie_id         object
processed_at     object
value           float64
title            object
dtype: object
Date range: 2023-01-02 → 2026-04-08
Nulls     :
source          0
dataset         0
serie_id        0
processed_at    0
value           0
title           0
dtype: int64
Duplicates: 0

── Silver — tiie_28 ──
Shape     : (819, 6)
Dtypes    :
source           object
dataset          object
serie_id         object
processed_at     object
value           float64
title            object
dtype: object
Date range: 2023-01-02 → 2026-04-09
Nulls     :
source          0
dataset         0
serie_id        0
processed_at    0
value           0
title           0
dtype: int64
Duplicates: 0

── Silver — inpc ──
Shape     : (38, 6)
Dtypes    :
source           object
dataset          object
serie_id         object
processed_at     object
value           float64
title            object


## 3. Monthly Aggregation

Aggregate each series to monthly frequency.

**Design decisions:**
- Daily series (FX, TIIE): group by month-start (`freq="MS"`) and aggregate
- Monthly series (INPC): already monthly — compute derived metrics directly
- NaN values from first-period calculations are **preserved** — not filled with 0.
  A NaN month-over-month change is semantically different from a 0% change.
  Tableau handles NaN gracefully as null/blank in charts.

In [108]:

def aggregate_fx(df: pd.DataFrame) -> pd.DataFrame:
        """
        Aggregate daily FX series to monthly.

        Metrics:
        - fx_rate      : monthly average USD/MXN
        - fx_volatility: monthly std deviation — proxy for exchange rate risk
        - fx_mom_pct   : month-over-month change (%) — first month is NaN by design
        """

        monthly = df.groupby(pd.Grouper(freq="MS")).agg(
        fx_rate      = ("value", "mean"),
        fx_volatility = ("value", "std"),
        )
        monthly["fx_mom_pct"] = monthly["fx_rate"].pct_change() * 100
        
        return monthly


fx_monthly = aggregate_fx(fx_raw)
inspect_df(fx_monthly, "FX Monthly")
print(fx_monthly.head(5))


── FX Monthly ──
Shape     : (40, 3)
Dtypes    :
fx_rate          float64
fx_volatility    float64
fx_mom_pct       float64
dtype: object
Date range: 2023-01-01 → 2026-04-01
Nulls     :
fx_rate          0
fx_volatility    0
fx_mom_pct       1
dtype: int64
Duplicates: 0
              fx_rate  fx_volatility  fx_mom_pct
date                                            
2023-01-01  18.986336       0.242484         NaN
2023-02-01  18.598568       0.226939   -2.042353
2023-03-01  18.374905       0.335185   -1.202587
2023-04-01  18.085494       0.082813   -1.575029
2023-05-01  17.737300       0.152637   -1.925269


In [109]:
def aggregate_tiie(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate daily TIIE 28-day series to monthly.

    Metrics:
    - tiie_28        : monthly average rate
    - tiie_mom_change: absolute change in basis points — first month is NaN by design

    Note: TIIE includes weekends with same value as preceding Friday.
    Monthly average naturally smooths this out.
    """
    monthly = df.groupby(pd.Grouper(freq="MS")).agg(
        tiie_28 = ("value", "mean")
    )
                                        
    monthly["tiie_mom_change"] = monthly["tiie_28"].diff()
    return monthly


tiie_monthly = aggregate_tiie(tiie_raw)
inspect_df(tiie_monthly, "TIIE Monthly")
print(tiie_monthly.head(5))


── TIIE Monthly ──
Shape     : (40, 2)
Dtypes    :
tiie_28            float64
tiie_mom_change    float64
dtype: object
Date range: 2023-01-01 → 2026-04-01
Nulls     :
tiie_28            0
tiie_mom_change    1
dtype: int64
Duplicates: 0
              tiie_28  tiie_mom_change
date                                  
2023-01-01  10.782818              NaN
2023-02-01  11.125763         0.342945
2023-03-01  11.347809         0.222046
2023-04-01  11.529950         0.182141
2023-05-01  11.533100         0.003150


In [110]:
def aggregate_inpc(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute derived inflation metrics from monthly INPC index.

    Metrics:
    - inpc        : raw index value (base year = varies by Banxico methodology)
    - inpc_mom_pct: month-over-month inflation (%) — first month is NaN by design
    - inpc_yoy_pct: year-over-year inflation (%) — first 12 months are NaN by design

    INPC lag: March and April 2026 data not yet published as of pipeline run date.
    Banxico publishes INPC ~10-15 days after month-end.
    """
    monthly = df[["value"]].copy()
    monthly = monthly.rename(columns={"value": "inpc"})
    monthly["inpc_mom_pct"] = monthly["inpc"].pct_change() * 100
    monthly["inpc_yoy_pct"] = monthly["inpc"].pct_change(periods=12) * 100
    return monthly


inpc_monthly = aggregate_inpc(inpc_raw)
inspect_df(inpc_monthly, "INPC Monthly")
print(inpc_monthly.head(5))


── INPC Monthly ──
Shape     : (38, 3)
Dtypes    :
inpc            float64
inpc_mom_pct    float64
inpc_yoy_pct    float64
dtype: object
Date range: 2023-01-01 → 2026-02-01
Nulls     :
inpc             0
inpc_mom_pct     1
inpc_yoy_pct    12
dtype: int64
Duplicates: 0
               inpc  inpc_mom_pct  inpc_yoy_pct
date                                           
2023-01-01  127.336           NaN           NaN
2023-02-01  128.046      0.557580           NaN
2023-03-01  128.389      0.267872           NaN
2023-04-01  128.363     -0.020251           NaN
2023-05-01  128.084     -0.217352           NaN


## 4. Build Gold DataFrame

Join the three monthly-aggregated series into a single Gold table.

**Join strategy: inner join**
- Only dates where all three series have data are included
- INPC has 38 months vs 40 for FX and TIIE — the 2 most recent months are excluded
- This is correct: we never want incomplete rows in Gold

**NaN policy:**
- First-period NaN values (mom_pct, yoy_pct) are preserved as null
- Gold consumers (Tableau) handle null gracefully
- Filling with 0 would imply "no change" which is semantically wrong

In [111]:
def build_gold_dataframe(
    fx: pd.DataFrame,
    tiie: pd.DataFrame,
    inpc: pd.DataFrame,
) -> pd.DataFrame:
    """
    Join the three monthly-aggregated series into a single Gold DataFrame.

    Uses inner join so only months where all three series have data are included.
    NaN values from first-period calculations are preserved — not filled with 0.

    Final schema:
        date           : datetime index (month-start)
        fx_rate        : float — monthly average USD/MXN
        fx_volatility  : float — monthly std deviation of daily FX
        fx_mom_pct     : float — FX month-over-month change (%)
        tiie_28        : float — monthly average TIIE 28-day rate
        tiie_mom_change: float — TIIE absolute change month-over-month
        inpc           : float — INPC index value
        inpc_mom_pct   : float — monthly inflation (%)
        inpc_yoy_pct   : float — year-over-year inflation (%)
    """
    df = fx.join(tiie, how="inner").join(inpc, how="inner")
    return df.sort_index()


gold = build_gold_dataframe(fx_monthly, tiie_monthly, inpc_monthly)

print(f"Shape     : {gold.shape}")
print(f"Date range: {gold.index.min().date()} → {gold.index.max().date()}")
print(f"\nNulls:\n{gold.isnull().sum()}")
print(f"\nSample:\n{gold.head(5)}")

Shape     : (38, 8)
Date range: 2023-01-01 → 2026-02-01

Nulls:
fx_rate             0
fx_volatility       0
fx_mom_pct          1
tiie_28             0
tiie_mom_change     1
inpc                0
inpc_mom_pct        1
inpc_yoy_pct       12
dtype: int64

Sample:
              fx_rate  fx_volatility  fx_mom_pct    tiie_28  tiie_mom_change  \
date                                                                           
2023-01-01  18.986336       0.242484         NaN  10.782818              NaN   
2023-02-01  18.598568       0.226939   -2.042353  11.125763         0.342945   
2023-03-01  18.374905       0.335185   -1.202587  11.347809         0.222046   
2023-04-01  18.085494       0.082813   -1.575029  11.529950         0.182141   
2023-05-01  17.737300       0.152637   -1.925269  11.533100         0.003150   

               inpc  inpc_mom_pct  inpc_yoy_pct  
date                                             
2023-01-01  127.336           NaN           NaN  
2023-02-01  128.046      0.

## 5. Data Quality Checks

Verify the Gold DataFrame before writing to S3.
These ranges inform the Great Expectations suite in P3.

In [112]:
print("── Value ranges ──")
print(gold.describe().round(4))

print("\n── Correlation matrix ──")
print(gold[["fx_rate", "tiie_28", "inpc"]].corr().round(3))

print("\n── Rows with any null ──")
null_rows = gold[gold.isnull().any(axis=1)]
print(f"Total: {len(null_rows)}")
print(null_rows)

── Value ranges ──
       fx_rate  fx_volatility  fx_mom_pct  tiie_28  tiie_mom_change      inpc  \
count  38.0000        38.0000     37.0000  38.0000          37.0000   38.0000   
mean   18.3768         0.1898     -0.2334  10.2302          -0.0945  135.5361   
std     1.1779         0.0935      2.4999   1.5005           0.1681    5.1682   
min    16.7918         0.0424     -3.9024   7.2865          -0.4803  127.3360   
25%    17.2578         0.1338     -1.8709   9.1069          -0.1812  130.8180   
50%    18.2961         0.1548     -0.5801  11.0031          -0.0680  136.0080   
75%    19.1211         0.2386      0.1111  11.4986           0.0002  139.9140   
max    20.5490         0.4442      8.4783  11.5331           0.3429  144.3070   

       inpc_mom_pct  inpc_yoy_pct  
count       37.0000       26.0000  
mean         0.3391        4.2395  
std          0.2641        0.5504  
min         -0.2174        3.5124  
25%          0.2343        3.7783  
50%          0.3136        4.2649  

## 6. Gold Layer — Design Decisions

| Decision | Observation | Action |
|---|---|---|
| Granularity | INPC is monthly — lowest common frequency | Aggregate all series to monthly |
| Join type | INPC has 2-month lag vs FX/TIIE | Inner join — exclude incomplete months |
| NaN policy | First-period pct_change returns NaN | Preserve as null — semantically correct |
| FX volatility | std of daily values within month | Proxy for exchange rate risk — relevant for fintech credit models |
| TIIE metric | Absolute change preferred over pct_change | Rate changes measured in basis points, not percent |
| INPC value | Raw index included alongside derived metrics | Allows Tableau to show index level + inflation rate |
| Partition key | year/month of data date | Consistent with Silver — enables Athena pruning |

## Next Steps
- [ ] Implement `gold.py` with `run_gold()` public interface
- [ ] Add Gold table to Glue Data Catalog via Terraform
- [ ] Register partitions with `MSCK REPAIR TABLE` after first run
- [ ] Connect Tableau Public to Athena Gold table